RAG PIPELINE - DATA INGECTION TO VECTOR DB PIPELINE

In [1]:
import os
from pathlib import Path
from IPython.display import display
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.config import Settings

C:\Users\hario\AppData\Local\Temp\ipykernel_10560\83065230.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\rag\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [85]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


# Function to read all PDF files from a directory
def read_all_pdfs(folder_path):

    # Create an empty list to store all documents/pages
    all_documents = []

    # Convert the folder path into a Path object
    pdf_dir = Path(folder_path)

    # Find all PDF files inside the directory and its subdirectories
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    # Display the number of PDF files found
    print(f"Found {len(pdf_files)} PDF files to process")

    # Process each PDF file one by one
    for pdf_file in pdf_files:

        # Display the name of the PDF currently being processed
        print(f"\nProcessing: {pdf_file.name}")

        try:
            # Create a PDF loader for the current PDF file
            loader = PyPDFLoader(str(pdf_file))

            # Load the PDF pages as LangChain Document objects
            documents = loader.load()

            # Add additional information to each document's metadata
            for doc in documents:

                # Store the PDF file name
                doc.metadata["source_file"] = pdf_file.name

                # Store the type of the file
                doc.metadata["file_type"] = "pdf"

            # Add the loaded documents to the main list
            all_documents.extend(documents)

        except Exception as e:
            # Display an error if a PDF cannot be processed
            print(f"Error processing {pdf_file.name}: {e}")

    # Display the total number of pages loaded
    print(f"\nSuccessfully loaded {len(all_documents)} pages")

    # Return all loaded documents
    return all_documents


# Call the function and read all PDFs from the data folder
all_pdf_documents = read_all_pdfs("../data")

Found 2 PDF files to process

Processing: cpi data for rag.pdf

Processing: GDP YEAR 24-25.pdf

Successfully loaded 21 pages


In [86]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-01-12T15:03:30+05:30', 'author': 'Mospi', 'moddate': '2026-01-12T15:49:42+05:30', 'source': '..\\data\\pdf\\cpi data for rag.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'cpi data for rag.pdf', 'file_type': 'pdf'}, page_content='This Press Release is embargoed against publication, telecast or circulation on internet till 4:00 PM of 12th January, 2026  \nPage 1 of 11 \n \n \n    \nPrice \nStatistics \nDivision \nIS/ ISO \n9001: 2015 \nCERTIFIED \nGOVERNMENT OF INDIA  \nMINISTRY OF STATISTICS AND  PROGRAMME IMPLEMENTATION  \nNATIONAL STATISTICS OFFICE \n \nDated 12th January, 2026 \n22 Pausa, Saka 1947 \nPRESS RELEASE \n \nCONSUMER PRICE INDEX NUMBERS ON BASE 2012=100 FOR RURAL, \nURBAN AND COMBINED FOR THE MONTH OF DECEMBER, 2025 \nI. Key highlights \n \n1. Headline Inflation: Year-on-year inflation rate based on All India Consumer \nPrice Index (CPI)

In [87]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [88]:
chunks = split_documents(all_pdf_documents)

Split 21 documents into 60 chunks

Example chunk:
Content: This Press Release is embargoed against publication, telecast or circulation on internet till 4:00 PM of 12th January, 2026  
Page 1 of 11 
 
 
    
Price 
Statistics 
Division 
IS/ ISO 
9001: 2015 
C...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-01-12T15:03:30+05:30', 'author': 'Mospi', 'moddate': '2026-01-12T15:49:42+05:30', 'source': '..\\data\\pdf\\cpi data for rag.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'cpi data for rag.pdf', 'file_type': 'pdf'}


In [89]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [90]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. "
                f"Embedding dimension: {self.model.get_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initializing the embadding manager

embedding_manager = EmbeddingManager()


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11262.67it/s]


Model loaded successfully. Embedding dimension: 384


vector db

In [91]:
# Extract text from chunks
texts = [chunk.page_content for chunk in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 60 texts...


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.12s/it]

Generated embeddings with shape: (60, 384)


In [92]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistent ChromaDB client
            os.makedirs(
                self.persist_directory,
                exist_ok=True
            )

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        # Check that documents and embeddings have the same length
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        # Process each document and embedding
        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Store document text
            documents_text.append(doc.page_content)

            # Convert NumPy array to Python list
            embeddings_list.append(embedding.tolist())

        # Add documents to ChromaDB
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Create vector store
vector_store = VectorStore()



Vector store initialized. Collection: pdf_documents
Existing documents in collection: 300


In [93]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-01-12T15:03:30+05:30', 'author': 'Mospi', 'moddate': '2026-01-12T15:49:42+05:30', 'source': '..\\data\\pdf\\cpi data for rag.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'cpi data for rag.pdf', 'file_type': 'pdf'}, page_content='This Press Release is embargoed against publication, telecast or circulation on internet till 4:00 PM of 12th January, 2026  \nPage 1 of 11 \n \n \n    \nPrice \nStatistics \nDivision \nIS/ ISO \n9001: 2015 \nCERTIFIED \nGOVERNMENT OF INDIA  \nMINISTRY OF STATISTICS AND  PROGRAMME IMPLEMENTATION  \nNATIONAL STATISTICS OFFICE \n \nDated 12th January, 2026 \n22 Pausa, Saka 1947 \nPRESS RELEASE \n \nCONSUMER PRICE INDEX NUMBERS ON BASE 2012=100 FOR RURAL, \nURBAN AND COMBINED FOR THE MONTH OF DECEMBER, 2025 \nI. Key highlights \n \n1. Headline Inflation: Year-on-year inflation rate based on All India Consumer \nPrice Index (CPI)

In [94]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vector_store.add_documents(chunks,embeddings)

Generating embeddings for 60 texts...


Batches: 100%|██████████| 2/2 [00:03<00:00,  1.87s/it]


Generated embeddings with shape: (60, 384)
Adding 60 documents to vector store...
Successfully added 60 documents to vector store
Total documents in collection: 360


Retriever Pipeline From VectorStore

In [95]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector store
        try:

            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # Convert L2 distance to a similarity score
                    similarity_score = 1 / (1 + distance)

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:

                print("No documents found")

            return retrieved_docs

        except Exception as e:

            print(f"Error during retrieval: {e}")

            return []


# Create RAG retriever
rag_retriever = RAGRetriever(
    vector_store,
    embedding_manager
)

In [96]:
results = rag_retriever.retrieve(
    "What is machine learning?",
    top_k=5
)

Retrieving documents for query: 'What is machine learning?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.79it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


In [97]:
rag_retriever

In [98]:
rag_retriever.retrieve("What is machine learning")

Retrieving documents for query: 'What is machine learning'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_b190fbb3_38',
  'content': 'This Press Release is embargoed against publication, telecast or circulation on internet till 4:00 pm on 7th January, 2025. \n \n2 \n \nI. Annual GDP Estimates and Growth Rates \n \n Real GDP or GDP at Constant Prices is estimated to attain a level of ₹ 184.88 lakh crore in \nthe financial year 2024-25, against the Provisional Estimate of GDP for the year 2023-24 of ₹173.82 \nlakh crore. The growth rate in Real GDP during 2024-25 is estimated at 6.4% as compared to 8.2% \nin 2023-24. Nominal GDP or GDP at Current Prices is estimated to attain a level of ₹ 324.11 lakh \ncrore in the year 2024-25, against ₹295.36 lakh crore in 2023-24, showing a growth rate of 9.7%.  \n Real GVA is estimated at ₹168.91 lakh crore in the year 2024-25, against the PE for the year \n2023-24 of ₹158.74 lakh crore, registering a growth rate of 6.4% as compared to 7.2% growth rate \nin 2023-24. Nominal GVA is estimated to attain a level of ₹2 92.64 lakh crore during FY 

In [99]:
rag_retriever.retrieve("explain me Response rate?")


Retrieving documents for query: 'explain me Response rate?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.69it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_748920fb_57',
  'content': 'Services 6,04,209 6,63,458 6,82,356 9.8 2.8\n2.3. Construction 21,78,693 23,83,877 25,88,257 9.4 8.6\n3. Tertiary Sector 1,33,60,808 1,46,43,960 1,61,83,120 9.6 10.5\n3.1 Trade, Hotels, Transport, Communication & \nServices related to Broadcasting 44,10,148 46,84,542 50,57,475 6.2 8.0\n3.2 Financial, Real Estate & Professional Services55,20,163 60,64,251 66,86,882 9.9 10.3\n3.3 Public Administration, Defence & Other \nServices* 34,30,497 38,95,167 44,38,763 13.5 14.0\nGVA at Basic Prices 2,46,59,041 2,67,62,147 2,92,63,609 8.5 9.3\nFRE: First Revised Estimates; PE: Provisional Estimates; FAE: First Advance Estimates\n*Public Administration, Defence & Other Services category includes the Other Services sector i.e. Education, Health, Recreation, and other \npersonal services\nStatement 4:  First Advance Estimates of GVA at Basic Prices by Economic Activity (at Current Prices)\nIndustry 2022-23\n(FRE)\nPercentage Change Over \nPrevious Year2023-24 

Integration Vector Context pipeline With LLM output

In [100]:
# ============================================================
# Simple RAG Pipeline with GPT
# ============================================================

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


# ============================================================
# 1. Load OpenAI API Key
# ============================================================

# .env is one folder above the notebook
load_dotenv("../.env")

openai_api_key = os.environ.get("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set")

print("OpenAI API key loaded successfully")


# ============================================================
# 2. Initialize GPT
# ============================================================

llm = ChatOpenAI(
    api_key=openai_api_key,
    model="gpt-4o-mini",
    temperature=0.1,
    max_tokens=1024
)

print("GPT LLM initialized successfully")


# ============================================================
# 3. RAG Retriever
# ============================================================

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")


        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]


        # Search ChromaDB
        try:

            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k
            )


            retrieved_docs = []


            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]


                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # Convert L2 distance to similarity
                    similarity_score = 1 / (1 + distance)


                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })


                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents"
                )


            else:

                print("No documents found")


            return retrieved_docs


        except Exception as e:

            print(f"Error during retrieval: {e}")

            return []


# ============================================================
# 4. Create RAG Retriever
# ============================================================

rag_retriever = RAGRetriever(
    vector_store,
    embedding_manager
)


# ============================================================
# 5. Simple RAG Function
# ============================================================

def rag_simple(
    query,
    retriever,
    llm,
    top_k=3
):

    # Retrieve documents
    results = retriever.retrieve(
        query,
        top_k=top_k
    )


    # Create context
    context = "\n\n".join(
        doc["content"]
        for doc in results
    )


    # Check context
    if not context:

        return "No relevant context found."


    # Create prompt
    prompt = f"""
You are a helpful AI assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Instructions:
- Use only the information from the context.
- If the answer is not present in the context, say:
  "The information is not available in the provided document."
- Give a clear and concise answer.

Answer:
"""


    # Generate answer using GPT
    try:

        response = llm.invoke(prompt)

        return response.content

    except Exception as e:

        return f"Error generating response: {str(e)}"


# ============================================================
# 6. Test RAG
# ============================================================

query = "Response rate?"


answer = rag_simple(
    query=query,
    retriever=rag_retriever,
    llm=llm,
    top_k=3
)


print("\nQuestion:")
print(query)


print("\nAnswer:")
print(answer)

OpenAI API key loaded successfully
GPT LLM initialized successfully
Retrieving documents for query: 'Response rate?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents

Question:
Response rate?

Answer:
Error generating response: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
